# M10.1 · Paralel mimariler, hızlanma yasaları ve paralel program tasarımı

Saat frekansı ~2005'te 2–4 GHz'de takılınca performansın tek yolu paralellik oldu: çok çekirdekli CPU'lar ve GPGPU. Flynn sınıflaması (SISD, SIMD, MISD, MIMD; paylaşımlı/dağıtık bellek, NUMA, efendi–işçi), heterojen mimariler (Cell BE, Kepler SMX, APU), Amdahl ve Gustafson yasaları, verimlilik ve Karp–Flatt ölçüsü, iş–yol (work–span) modeli ve Brent teoremi. Tasarım: Foster'ın PCAM'i (bölümleme, iletişim, birleştirme, eşleme), ayrıştırma desenleri (görev paralelliği, böl ve yönet, geometrik, özyinelemeli veri, boru hattı, olay tabanlı) ve program yapısı desenleri (SPMD, MPMD, efendi–işçi, fork–join, döngü paralelliği). Saf Python'da liste zamanlama ve boru hattı simülasyonu, NumPy'da kritik yol ve Amdahl uydurma, C++ `std::thread` ile Mandelbrot'ta statik, döngüsel ve dinamik yük dağıtımı.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m10-1-parallel-models.html](https://m-gonul.github.io/pytorch-lab/m10-1-parallel-models.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m10_1_parallel_models.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef rand_dag(n, seed):\n    rng = random.Random(seed)\n    dur = {i: rng.randint(1, 9) for i in range(n)}\n    preds = {i: [j for j in range(max(0, i - 6), i) if rng.random() < 0.3] for i in range(n)}\n    return dur, preds\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def laws():\n        close(need(ns, \"amdahl\")(0.9, 8), 1 / 0.2125, what=\"Amdahl\")\n        close(need(ns, \"gustafson\")(0.9, 8), 7.3, what=\"Gustafson\")\n        close(need(ns, \"karp_flatt\")(ns[\"amdahl\"](0.95, 16), 16), 0.05, what=\"Karp–Flatt Amdahl'ı geri verir\")\n        close(need(ns, \"pipeline_makespan\")([2, 3, 5, 2, 3], 100), 510, what=\"boru hattı\")\n        close(ns[\"pipeline_makespan\"]([1, 1, 1], 1), 3, what=\"tek öğe\")\n    R_.test(\"yasalar ve boru hattı\", laws)\n\n    def sched():\n        rows = []\n        for n, seed in ((30, 0), (80, 1), (200, 2)):\n            dur, preds = rand_dag(n, seed)\n            W, S = need(ns, \"work_span\")(dur, preds)\n            ensure(W == sum(dur.values()), \"iş = süre toplamı\")\n            r = [n, W, S, f\"{W / S:.1f}\"]\n            for p in (1, 2, 4, 8):\n                T = need(ns, \"list_schedule\")(dur, preds, p)\n                ensure(max(W / p, S) - 1e-9 <= T <= W / p + S + 1e-9, f\"Brent sınırları (n = {n}, p = {p})\")\n                if p == 1:\n                    ensure(T == W, \"tek işlemcide T = W\")\n                r.append(f\"{T:g}\")\n            rows.append(r)\n        R_.table(\"Rastgele görev grafikleri, alt-seviye öncelikli liste zamanlama\", [\"görev\", \"W\", \"S∞\", \"W/S∞\", \"T₁\", \"T₂\", \"T₄\", \"T₈\"], rows,\n                 note=\"p, paralellik W/S∞'u aştığında hızlanma doyar: kritik yol sınırı. Brent'in üst sınırı her açgözlü zamanlayıcı için geçerlidir.\")\n    R_.test(\"iş–yol ve zamanlama\", sched)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngü kuralları\", lambda: (max_loops(src, 1, func=\"critical_path\"), max_loops(src, 0, func=\"fit_amdahl\"), max_loops(src, 0, func=\"speedup_grid\")))\n\n    def cp():\n        rng = np.random.default_rng(0)\n        n = 60\n        A = np.triu((rng.random((n, n)) < 0.08).astype(float), 1)\n        dur = rng.integers(1, 10, n).astype(float)\n        ef, S = need(ns, \"critical_path\")(A, dur)\n        ref = np.zeros(n)\n        for j in range(n):\n            ref[j] = dur[j] + max([ref[i] for i in range(j) if A[i, j]], default=0)\n        close(ef, ref, what=\"en erken bitişler\")\n        close(S, ref.max(), what=\"kritik yol\")\n    R_.test(\"critical_path\", cp)\n\n    def fit():\n        p = np.array([1, 2, 4, 8, 16, 32.0])\n        T = 10 * (0.07 + 0.93 / p) * (1 + 0.01 * np.random.default_rng(1).normal(size=6))\n        s, T1 = need(ns, \"fit_amdahl\")(p, T)\n        ensure(abs(s - 0.07) < 0.01 and abs(T1 - 10) < 0.3, f\"seri oran ≈ 0.07 (bulunan {s:.3f})\")\n        am, gu = need(ns, \"speedup_grid\")(np.array([0.5, 0.9, 0.99, 0.999]), np.array([4, 16, 64, 1024.0]))\n        close(am[1, 0], 1 / (0.1 + 0.9 / 4), what=\"Amdahl tablosu\")\n        R_.table(\"Amdahl hızlanması (Gustafson parantezde)\", [\"f\", \"p = 4\", \"16\", \"64\", \"1024\"],\n                 [[f, *[f\"{am[i, j]:.1f} ({gu[i, j]:.0f})\" for j in range(4)]] for i, f in enumerate([0.5, 0.9, 0.99, 0.999])],\n                 note=\"Binlerce çekirdekten yararlanmak için seri oran binde birin altında olmalı (Amdahl) — ya da problem çekirdeklerle büyümeli (Gustafson). GPU'lar ikinci dünyanın aracıdır.\")\n    R_.test(\"Amdahl uydurma ve tablolar\", fit)\n",
 "d": "import os\nimport time\nimport torch\nfrom labcheck import close, need, ensure, fmt_time\n\n\ndef ref_mandel(H, W, maxit):\n    c = torch.linspace(0, W - 1, W, dtype=torch.float64)\n    r = torch.linspace(0, H - 1, H, dtype=torch.float64)\n    cr = (-2.0 + 3.0 * c / (W - 1))[None].expand(H, W)\n    ci = (1.5 * r / (H - 1))[:, None].expand(H, W)\n    zr = torch.zeros(H, W, dtype=torch.float64)\n    zi = torch.zeros_like(zr)\n    k = torch.zeros(H, W, dtype=torch.int32)\n    for _ in range(maxit):\n        alive = zr * zr + zi * zi <= 4.0\n        if not alive.any():\n            break\n        t = zr * zr - zi * zi + cr\n        zi = torch.where(alive, 2 * zr * zi + ci, zi)\n        zr = torch.where(alive, t, zr)\n        k += alive.int()\n    return k\n\n\ndef check(ns, R_):\n    ext = need(ns, \"build\")()\n\n    def corr():\n        ref = ref_mandel(120, 180, 300)\n        for p in (1, 3):\n            for m in (\"block\", \"cyclic\", \"dynamic\"):\n                img, work = ext.mandelbrot(120, 180, 300, p, m)\n                ensure(img.dtype == torch.int32 and tuple(work.shape) == (p,), \"dönüş tipleri\")\n                agree = float((img == ref).double().mean())\n                ensure(agree > 0.999, f\"{m}/{p}: referansla uyum {agree:.4f}\")\n                ensure(int(work.sum()) == int(img.long().sum()), \"iş toplamı = iterasyon toplamı\")\n    R_.test(\"doğruluk\", corr)\n\n    def bal():\n        H, W, it = 600, 900, 1000\n        p = max(2, min(8, os.cpu_count() or 2))\n        rows = []\n        for m in (\"block\", \"cyclic\", \"dynamic\"):\n            best = 1e9\n            for _ in range(3):\n                t0 = time.perf_counter()\n                _, work = ext.mandelbrot(H, W, it, p, m)\n                best = min(best, time.perf_counter() - t0)\n            imb = float(work.max()) / float(work.double().mean())\n            rows.append([m, fmt_time(best), f\"{imb:.2f}\", \" \".join(f\"{float(x) / 1e6:.0f}M\" for x in work)])\n            if m == \"block\":\n                imb_block = imb\n            if m == \"cyclic\":\n                imb_cyc = imb\n        t0 = time.perf_counter()\n        ext.mandelbrot(H, W, it, 1, \"block\")\n        t1 = time.perf_counter() - t0\n        ensure(imb_block > 1.2 and imb_cyc < 1.1, \"blok dağıtım dengesiz, döngüsel dengeli olmalı\")\n        R_.table(f\"Mandelbrot {H}×{W}, maxit {it}, {p} thread (1 thread: {fmt_time(t1)})\", [\"dağıtım\", \"süre\", \"en çok iş / ortalama\", \"thread başına iterasyon\"], rows,\n                 note=\"Üst yarı düzlemde kümenin içi (pahalı satırlar) eksene yakın: ardışık bloklarda ilk thread'in işi çok fazla. Döngüsel dağıtım satırları serpiştirir; dinamik dağıtım hiçbir maliyet modeli bilmeden dengeler ama her satırda atomik sayaç maliyeti öder.\")\n    R_.test(\"yük dengesi\", bal)\n",
 "e": "import os\nimport time\nimport hashlib\nimport torch\nfrom torch.utils.cpp_extension import load_inline\nfrom labcheck import close, need, ensure, bench, fmt_time\n\nSRC = r\"\"\"\n#include <torch/extension.h>\n#include <thread>\n#include <atomic>\n#include <vector>\n#include <tuple>\nstatic int esc(double cr, double ci, int m) { double zr = 0, zi = 0; int k = 0; while (k < m && zr * zr + zi * zi <= 4.0) { double t = zr * zr - zi * zi + cr; zi = 2 * zr * zi + ci; zr = t; ++k; } return k; }\nstd::tuple<torch::Tensor, torch::Tensor> mandelbrot(int64_t H, int64_t W, int64_t maxit, int64_t nt, std::string mode) {\n    auto out = torch::zeros({H, W}, torch::kInt32); auto work = torch::zeros({nt}, torch::kInt64);\n    int32_t* p = out.data_ptr<int32_t>(); std::atomic<int64_t> nx(0); std::vector<std::thread> ts;\n    for (int64_t t = 0; t < nt; ++t) ts.emplace_back([=, &nx]() { for (int64_t r = nx.fetch_add(1); r < H; r = nx.fetch_add(1)) { double ci = 1.5 * r / (H - 1); for (int64_t c = 0; c < W; ++c) p[r * W + c] = esc(-2.0 + 3.0 * c / (W - 1), ci, maxit); } });\n    for (auto& th : ts) th.join();\n    return {out, work};\n}\n\"\"\"\n\n\ndef py_mandel(H, W, maxit):\n    out = [[0] * W for _ in range(H)]\n    for r in range(H):\n        ci = 1.5 * r / (H - 1)\n        for c in range(W):\n            cr = -2.0 + 3.0 * c / (W - 1)\n            zr = zi = 0.0\n            k = 0\n            while k < maxit and zr * zr + zi * zi <= 4.0:\n                zr, zi = zr * zr - zi * zi + cr, 2 * zr * zi + ci\n                k += 1\n            out[r][c] = k\n    return out\n\n\ndef check(ns, R_):\n    ext = load_inline(name=\"ptl_m101e_\" + hashlib.md5(SRC.encode()).hexdigest()[:8], cpp_sources=SRC, functions=[\"mandelbrot\"], extra_cflags=[\"-O2\"], verbose=False)\n\n    def t():\n        H, W, it = 300, 450, 500\n        mt = need(ns, \"mandel_torch\")(H, W, it)\n        mc, _ = ext.mandelbrot(H, W, it, 1, \"dynamic\")\n        ensure(float((mt == mc).double().mean()) > 0.999, \"torch sürümü C++ ile aynı\")\n        ncpu = os.cpu_count() or 2\n        threads = sorted({1, 2, max(2, ncpu // 2), ncpu})\n        T, S = need(ns, \"scaling\")(ext, 600, 900, 1000, threads)\n        s, T1 = need(ns, \"fit_amdahl\")(threads, T)\n        t_torch = bench(lambda: ns[\"mandel_torch\"](H, W, it), repeat=1, number=1)\n        t_cpp = bench(lambda: ext.mandelbrot(H, W, it, ncpu, \"dynamic\"), repeat=3, number=1)\n        t_py = bench(lambda: py_mandel(30, 45, it), repeat=1, number=1) * 100\n        rows = [[f\"C++ dinamik, {p} thread (600×900, maxit 1000)\", fmt_time(T[i]), f\"{S[i]:.2f}×\"] for i, p in enumerate(threads)]\n        rows += [[f\"saf Python ({H}×{W}, tahmini)\", fmt_time(t_py), \"\"], [f\"PyTorch vektörel ({H}×{W})\", fmt_time(t_torch), \"\"], [f\"C++ {ncpu} thread ({H}×{W})\", fmt_time(t_cpp), \"\"]]\n        R_.table(f\"Mandelbrot ölçekleme ({ncpu} mantıksal çekirdek)\", [\"uygulama\", \"süre\", \"hızlanma\"], rows,\n                 note=f\"Amdahl uydurması: seri oran ≈ {max(s, 0):.3f}. Vektörel PyTorch, kaçmış pikselleri de her iterasyonda hesaplar (maske): iş verimsiz ama Python döngüsü yok. C++'ta her piksel yalnız kendi iterasyonu kadar çalışır.\")\n        ensure(S[-1] > 1.2 or ncpu < 2, \"çok thread hızlandırmalı\")\n    R_.test(\"ölçekleme\", t)\n",
 "x1": "import math\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows = []\n        for n in (8, 64, 1024):\n            dur, preds = need(ns, \"mergesort_dag\")(n)\n            W = sum(dur.values())\n            memo = {}\n\n            def ef(v):\n                if v not in memo:\n                    memo[v] = dur[v] + max((ef(u) for u in preds[v]), default=0)\n                return memo[v]\n\n            S = max(ef(v) for v in preds)\n            ensure(W == n + n * int(math.log2(n)), \"iş = n + n log n\")\n            ensure(S == 1 + 2 * n - 2, \"yol = 1 + (n + n/2 + … + 2)\")\n            mc = need(ns, \"max_concurrency\")(dur, preds)\n            ensure(mc == n, \"en çok eşzamanlılık: n yaprak\")\n            rows.append([n, W, S, f\"{W / S:.1f}\", mc])\n        R_.table(\"Üstten aşağı merge sort (birleştirme sıralı)\", [\"n\", \"iş W\", \"yol S∞\", \"paralellik\", \"en çok eşzamanlı görev\"], rows,\n                 note=\"Birleştirme adımı sıralı kaldıkça yol O(n): paralellik yalnız O(log n). Paralel birleştirme (ikili aramayla bölme) yolu O(log² n)'ye indirir.\")\n    R_.test(\"merge sort grafiği\", t)\n",
 "x2": "from labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        st = [2, 3, 5, 2, 3]\n        close(need(ns, \"simulate\")(st, [1] * 5, 100), 510, what=\"çoğaltmasız = el hesabı\")\n        rows = []\n        for rep in ([1, 1, 1, 1, 1], [1, 1, 2, 1, 1], [1, 2, 2, 1, 2], [1, 2, 3, 1, 2]):\n            T = ns[\"simulate\"](st, rep, 1000)\n            thr = need(ns, \"throughput\")(st, rep)\n            ensure(abs(1000 / T - thr) / thr < 0.02, \"uzun akışta simülasyon = darboğaz formülü\")\n            rows.append([str(rep), sum(rep), f\"{T:.0f}\", f\"{thr:.3f}\"])\n        R_.table(\"Boru hattı (2, 3, 5, 2, 3), 1000 öğe\", [\"çoğaltma\", \"işçi\", \"toplam süre\", \"verim (öğe/zaman)\"], rows,\n                 note=\"Verim en yavaş (çoğaltılmış) aşamaya bağlı: 5'lik aşamayı ikiye bölmek verimi 0.2'den 0.33'e çıkarır, artık darboğaz 3'lük aşamalardır.\")\n    R_.test(\"aşama çoğaltma\", t)\n",
 "x3": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows = []\n        for lam, mu, c in [(0.8, 1.0, 1), (1.8, 1.0, 2), (3.5, 1.0, 4), (7.0, 1.0, 8)]:\n            sim = sum(need(ns, \"simulate\")(lam, mu, c, 30000, s) for s in range(3)) / 3\n            th = need(ns, \"erlang_c_wait\")(lam, mu, c)\n            ensure(abs(sim - th) / th < 0.15, f\"simülasyon ≈ Erlang C (c = {c}: {sim:.2f} vs {th:.2f})\")\n            rows.append([c, lam, f\"{lam / (c * mu):.2f}\", f\"{th:.3f}\", f\"{sim:.3f}\"])\n        R_.table(\"M/M/c kuyruğu, ortalama bekleme\", [\"gişe c\", \"λ\", \"yük ρ\", \"Erlang C\", \"simülasyon\"], rows,\n                 note=\"Aynı yükte (ρ ≈ 0.9) gişe sayısı arttıkça bekleme düşer: havuzlanmış kaynaklar dalgalanmaları emer — iş havuzlu (dinamik) zamanlamanın kuyruk teorisi karşılığı. Olay kuyruğu düzensiz, önceden bilinmeyen iletişimi modellemenin klasik yoludur.\")\n    R_.test(\"ayrık olay simülasyonu\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m10_1_parallel_models as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Neden paralellik:** Moore (1970'ler): yonga başına transistör sayısı ~2 yılda bir ikiye katlanır. Uzun süre frekans da arttı; ısı ve güç yüzünden ~2004–2005'te 2–4 GHz'de doydu. Artan transistörler artık çekirdeğe (IBM Power4 2001: ilk çift çekirdek) ve GPU'ya gidiyor. GPU çekirdeği tek başına zayıf ama sayıca çok: hem GFLOP/s'de hem **GFLOP/W**'de (enerji başına hesap) üstün. Bedeli: paralel kod — algoritmalar yeniden tasarlanmalı, yeni API'ler (CUDA, OpenCL).

**Flynn (1966):** SISD (sıralı), SIMD (bir komut çok veri: vektör birimleri, GPU'nun SIMT'si), MISD (pratikte yok), MIMD (en esnek). MIMD → **paylaşımlı bellek** (tek adres uzayı; ölçeklenince NUMA: yerel bellek hızlı, uzak bellek yavaş; simetrik ya da efendi–işçi — GPU'lu sistemler bu sonuncusuna benzer) ve **dağıtık bellek** (mesaj geçirme: kümeler, süperbilgisayarlar, veri merkezlerinde büyük model eğitimi). CPU: önbellek ve kontrol mantığı alanı kaplar; GPU: hesap birimleri. Ayrık GPU'da PCIe veri aktarımı bir **eşik boyut** yaratır; bu yüzden CPU+GPU'nun aynı belleği paylaştığı heterojen tasarımlar (Cell BE, AMD APU, Kepler SMX'leri, Xeon Phi).

**Hızlanma ölçüleri:** $S=T_1/T_p$, verimlilik $E=S/p$.

- **Amdahl:** seri oran $1-f$ sabit, problem sabit → $S=1/((1-f)+f/N)\le1/(1-f)$.
- **Gustafson:** p işlemciyle zaman sabit, problem büyür → $S=(1-f)+fN$.
- **Karp–Flatt:** ölçülen S'den deneysel seri oran $e=(1/S-1/p)/(1-1/p)$; p ile artıyorsa ek yük (iletişim, dengesizlik) var.
- **İş–yol:** iş W (tek işlemcide süre), yol $S_\infty$ (kritik yol), paralellik $W/S_\infty$. Açgözlü zamanlama (Brent): $\max(W/p,S_\infty)\le T_p\le W/p+S_\infty$.

**Tasarım (Foster'ın PCAM'i):** **Bölümleme** — işlevsel (görev) ya da veri (alan) ayrıştırması; işlemci sayısının 1–2 mertebe fazlası kadar görev. **İletişim** — görev grafiği; veri bağımlılıkları. **Birleştirme** — iletişimi yok etmek için görevleri grupla (yine işlemciden fazla grup). **Eşleme** — yükü dengele, pahalı iletişimi aynı düğüme koy (NP-tam; sezgisel). Hesap tekrarlamak iletişimden ucuz olabilir.

**Ayrıştırma desenleri** (Mattson): görev paralelliği, böl ve yönet (merge sort), geometrik ayrıştırma (dizi/ızgara blokları + kenar "halo"), özyinelemeli veri (kısmi toplamlar: giderek büyüyen aralıklar), boru hattı (aşama süreleri dengeli olmalı), olay tabanlı koordinasyon (düzensiz iletişim, ayrık olay simülasyonu). **Program yapısı desenleri:** küresel paralel–yerel sıralı (SPMD, MPMD, efendi–işçi, map-reduce) ve küresel sıralı–yerel paralel (fork–join, döngü paralelliği).

## 1 · Elle

Amdahl: $1/(0.1+0.9/8)=1/0.2125=4.706$; $E=0.588$. Gustafson: 8 işlemcide aynı sürede 7.3 kat iş. Karp–Flatt: $(0.2125-0.125)/0.875=0.1$ — ölçüm tam Amdahl'a uyuyorsa e sabittir.

Boru hattı: ilk öğe tüm aşamalardan geçer (15), sonra her öğe darboğaz aşamanın süresi (5) kadar sonra çıkar: $15+99\cdot5$. Darboğaz dışındaki aşamalar zamanın çoğunda boştadır.

Ağaç toplama: her toplama bir sayıyı "yok eder" → $n-1$ iş; her seviye yarıya indirir → $\log_2n$ yol.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Yasalar: paralelleşebilir oran f = 0.9, N = 8**

- `am`: Amdahl hızlanması
- `lim`: N → ∞ sınırı
- `gu`: Gustafson (ölçeklenmiş) hızlanması
- `ef`: Amdahl verimliliği S/N
- `kf`: Karp–Flatt: ölçülen S = 4.70588, p = 8 → seri oran e

**Boru hattı ve indirgeme**

- `pipe`: Aşama süreleri (2, 3, 5, 2, 3), 100 öğe: toplam süre
- `wk`: 1024 sayının ağaç toplamı: iş (toplama sayısı)
- `sp`: Aynı ağacın yolu (adım sayısı)

In [ ]:
cevaplar = dict(
    am=None,
    lim=None,
    gu=None,
    ef=None,
    kf=None,
    pipe=None,
    wk=None,
    sp=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"am":[4.705882352941177,0.000001],"lim":[10,0],"gu":[7.3,0],"ef":[0.5882352941176471,0.000001],"kf":[0.1,0.0001],"pipe":[510,0],"wk":[1023,0],"sp":[10,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `amdahl(f, n)`, `gustafson(f, n)`, `karp_flatt(S, p)`.
- `pipeline_makespan(stages, n_items)` — her öğe aşamaları sırayla geçer, her aşama aynı anda tek öğe işler: $\text{bitiş}[i][s]=\max(\text{bitiş}[i][s-1],\text{bitiş}[i-1][s])+t_s$; son öğenin bitişi.
- `work_span(dur, preds)` — `dur` görev → süre sözlüğü, `preds` görev → öncül listesi; `(W, S∞)`.
- `list_schedule(dur, preds, p)` — olay güdümlü açgözlü zamanlama: hazır görevler arasından **alt seviyesi** (görevden çıkışa en uzun yol, kendi süresi dahil) en büyük olanı boş işlemciye ata; makespan döndür.

In [ ]:
%%writefile stage_b.py
def amdahl(f, n):
    ...


def gustafson(f, n):
    ...


def karp_flatt(S, p):
    ...


def pipeline_makespan(stages, n_items):
    ...


def work_span(dur, preds):
    ...


def list_schedule(dur, preds, p):
    ...


dur = {"a": 2, "b": 3, "c": 1, "d": 4}
preds = {"a": [], "b": ["a"], "c": ["a"], "d": ["b", "c"]}
print(work_span(dur, preds), list_schedule(dur, preds, 2))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy.

- `critical_path(A, dur)` — `A (n, n)` bitişiklik ($A_{ij}=1$: i → j), `dur (n,)`; en erken bitiş zamanları $\text{ef}_j=\text{dur}_j+\max_{i\to j}\text{ef}_i$ ("max-artı" çarpımıyla n kez gevşetme, tek döngü); `(ef, max(ef))`.
- `fit_amdahl(p, T)` — $T(p)=a+b/p$ en küçük kareler → `(seri oran a/(a+b), T1 = a+b)`.
- `speedup_grid(f, p)` — `f (F,)`, `p (P,)` → Amdahl ve Gustafson `(F, P)` tabloları, döngüsüz.

In [ ]:
%%writefile stage_c.py
import numpy as np


def critical_path(A, dur):
    ...


def fit_amdahl(p, T):
    ...


def speedup_grid(f, p):
    ...


A = np.zeros((4, 4)); A[0, 1] = A[0, 2] = A[1, 3] = A[2, 3] = 1
print(critical_path(A, np.array([2.0, 3, 1, 4])))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

C++ `std::thread` (`load_inline`). **Mandelbrot** $H\times W$: piksel (r, c) → $c_0=(-2+3c/(W-1))+i\,(1.5\,r/(H-1))$ (yalnız üst yarı: satır maliyetleri çok dengesiz), `maxit` sınırlı kaçış iterasyonu.

- `SRC` — `std::tuple<torch::Tensor, torch::Tensor> mandelbrot(int64_t H, int64_t W, int64_t maxit, int64_t nthreads, std::string mode)` → (iterasyon sayıları `int32 (H, W)`, thread başına toplam iterasyon `int64 (nthreads,)`). `mode`: `"block"` (ardışık satır blokları), `"cyclic"` (satır r → thread r mod p), `"dynamic"` (`std::atomic<int>` sayaçtan sıradaki satırı al).
- `build()` → modül.

In [ ]:
%%writefile stage_d.py
import hashlib
import torch
from torch.utils.cpp_extension import load_inline

SRC = r"""
#include <torch/extension.h>
#include <thread>
#include <atomic>
#include <vector>
#include <tuple>

std::tuple<torch::Tensor, torch::Tensor> mandelbrot(int64_t H, int64_t W, int64_t maxit, int64_t nthreads, std::string mode) {
    // TODO: nthreads tane std::thread başlat, satırları mode'a göre dağıt, join et.
    return {torch::zeros({H, W}, torch::kInt32), torch::zeros({nthreads}, torch::kInt64)};
}
"""


def build():
    name = "ptl_m101_" + hashlib.md5(SRC.encode()).hexdigest()[:8]
    return load_inline(name=name, cpp_sources=SRC, functions=["mandelbrot"], extra_cflags=["-O2"], verbose=False)


ext = build()
img, work = ext.mandelbrot(100, 150, 200, 2, "dynamic")
print(img.float().mean(), work)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

- `mandel_torch(H, W, maxit)` — PyTorch vektörel (maske ile; döngü yalnız iterasyonlar).
- `scaling(ext, H, W, maxit, threads)` → `(süreler, hızlanmalar)`; dinamik mod, her biri 3 tekrarın en iyisi.
- `fit_amdahl(p, T)` — c'deki gibi (NumPy).

In [ ]:
%%writefile stage_e.py
import os
import time
import hashlib
import numpy as np
import torch
from torch.utils.cpp_extension import load_inline


def mandel_torch(H, W, maxit):
    ...


def scaling(ext, H, W, maxit, threads):
    ...


def fit_amdahl(p, T):
    ...


print(mandel_torch(50, 75, 100).float().mean())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Böl ve yönet: merge sort bağımlılık grafiği

- `mergesort_dag(n)` — n (2'nin kuvveti) elemanlı üstten aşağı merge sort'un bağımlılık grafiği: düğümler `("sort", lo, hi)` ve `("merge", lo, hi)`; `sort` yaprakları 1 eleman (süre 1), `merge(lo, hi)` süresi `hi - lo`, `merge` iki alt `sort`'a bağlı, `sort(lo, hi)` kendi `merge`'üne eşit (süre 0). `(dur, preds)`.
- `max_concurrency(dur, preds)` — sınırsız işlemcide (ASAP) aynı anda çalışan en çok görev.

In [ ]:
%%writefile extra_x1.py
def mergesort_dag(n):
    ...


def max_concurrency(dur, preds):
    ...


d, p = mergesort_dag(8)
print(len(d), max_concurrency(d, p))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Dengesiz boru hattı ve aşama çoğaltma

- `simulate(stages, replicas, n_items)` — her aşamada `replicas[s]` özdeş işçi; öğeler sırayla; boşalan ilk işçiye. Toplam süre.
- `throughput(stages, replicas)` → $1/\max_s(t_s/r_s)$.

In [ ]:
%%writefile extra_x2.py
import heapq


def simulate(stages, replicas, n_items):
    ...


def throughput(stages, replicas):
    ...


print(simulate([2, 3, 5, 2, 3], [1, 1, 2, 1, 1], 100))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Olay tabanlı ayrıştırma: ayrık olay simülasyonu

**Banka kuyruğu** (M/M/c): müşteriler Poisson(λ) gelir, hizmet üstel(μ), c gişe. `heapq` olay kuyruğuyla ayrık olay simülasyonu.

- `simulate(lam, mu, c, n_customers, seed=0)` → ortalama bekleme.
- `erlang_c_wait(lam, mu, c)` — analitik $W_q=\dfrac{C(c,a)}{c\mu-\lambda}$, $a=\lambda/\mu$.

In [ ]:
%%writefile extra_x3.py
import heapq
import math
import random


def simulate(lam, mu, c, n_customers, seed=0):
    ...


def erlang_c_wait(lam, mu, c):
    ...


print(simulate(1.8, 1.0, 2, 20000), erlang_c_wait(1.8, 1.0, 2))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Gustafson (1988): Amdahl yasasını yeniden değerlendirmek

[Gustafson — Reevaluating Amdahl's Law (CACM 1988)](https://doi.org/10.1145/42411.42415)

d aşamasındaki C++ Mandelbrot'u (ya da M10.3'teki konvolüsyonu) kullanarak kendi makinende (i) sabit boyutta 1..p thread ile güçlü ölçekleme, (ii) thread sayısıyla orantılı büyüyen problemle zayıf ölçekleme ölç. c aşamasındaki Amdahl uydurmasıyla seri oranı kestir; Gustafson'un ölçeklenmiş hızlanma eğrisini çiz ve makaledeki 1024 işlemcili sonuçlarla nitel karşılaştır.


**Adımlar**

1. Her ölçümü 5 kez tekrarla, ortancayı al; ısınma (warm-up) çalıştırması ekle.
2. Hiper-iş parçacığı (SMT) olan çekirdeklerde eğrinin kırılmasını yorumla.

**Kabul ölçütleri**

- Güçlü ölçeklemede hızlanma Amdahl eğrisine, zayıf ölçeklemede Gustafson doğrusuna yakın.

## Makale ödevi: Blumofe & Leiserson (1999): İş çalma ile zamanlama

[Blumofe & Leiserson — Scheduling Multithreaded Computations by Work Stealing (JACM 1999)](https://doi.org/10.1145/324133.324234)

C++'ta thread başına çift uçlu kuyruk (deque) ile iş çalan bir zamanlayıcı yaz: sahibi alttan push/pop, hırsızlar üstten çalar. Paralel merge sort ve Fibonacci (fork–join) ile merkezi tek kuyruklu havuza karşı ölç; makaledeki $T_p\le T_1/p+O(T_\infty)$ sınırını b aşamasındaki iş–yol hesabıyla karşılaştır.


**Adımlar**

1. Önce mutex korumalı deque, sonra (isteğe bağlı) Chase–Lev kilitsiz deque.
2. Görev ayrıntı düzeyini (cutoff) tara.

**Kabul ölçütleri**

- Küçük görevlerde merkezi kuyruk çekişmeden yavaşlar, iş çalma ölçeklenir.
- Çalma sayısı p·T∞ mertebesinde.

## Kaynaklar

- [Barlas — Multicore and GPU Programming: An Integrated Approach (2. baskı), Bölüm 1–2](https://www.elsevier.com/books/multicore-and-gpu-programming/barlas/978-0-12-814120-5) — ©
- [Foster — Designing and Building Parallel Programs (1995, çevrimiçi)](https://www.mcs.anl.gov/~itf/dbpp/) — ücretsiz çevrimiçi
- [Amdahl — Validity of the single processor approach to achieving large scale computing capabilities (AFIPS 1967)](https://doi.org/10.1145/1465482.1465560) — ©
- [Gustafson — Reevaluating Amdahl's Law (CACM 1988)](https://doi.org/10.1145/42411.42415) — ©
- [Mattson, Sanders, Massingill — Patterns for Parallel Programming (2004)](https://www.oreilly.com/library/view/patterns-for-parallel/0321228111/) — ©
- [Blumofe & Leiserson — Scheduling Multithreaded Computations by Work Stealing (JACM 1999)](https://doi.org/10.1145/324133.324234) — ©